In [10]:
import requests
import json
from google.colab import userdata

API_KEY = userdata.get('YANDEX_API_KEY')
FOLDER_ID = userdata.get('YANDEX_FOLDER_ID')

def ask_yandex_gpt(system_prompt, user_prompt, temperature=0.3):
    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Api-Key {API_KEY}"
    }
    data = {
        "modelUri": f"gpt://{FOLDER_ID}/yandexgpt-lite/latest",
        "completionOptions": {
            "stream": False,
            "temperature": temperature,
            "maxTokens": 2000
        },
        "messages": [
            {"role": "system", "text": system_prompt},
            {"role": "user", "text": user_prompt}
        ]
    }
    response = requests.post(url, headers=headers, json=data)
    return response.json()['result']['alternatives'][0]['message']['text']

In [12]:

import json
import time
import re
import os

drive_path = '/content/drive/MyDrive/research_proposal'
os.makedirs(drive_path, exist_ok=True)
save_file = f'{drive_path}/synthetic_tasks.json'

synthetic_dataset = []

system_prompt_gen = """Ты — эксперт по составлению задач по программированию на Python.
Отвечай СТРОГО в формате JSON, представляющем массив из 5 объектов. Никакого лишнего текста, вводных слов или пояснений.
Формат каждого объекта:
{
  "id": "уникальный строковый номер (например task_1)",
  "instruction": "условие задачи",
  "evaluate_code": "строка с кодом Python функции тестирования (например, def evaluate(solution): ...)",
  "example solution": "строка с кодом правильного решения задачи",
  "failure_cases": ["строка с кодом неправильного решения 1", "строка с кодом неправильного решения 2"],
}"""


user_prompt_gen = """Сгенерируй 5 задачи по программированию на Python.
Верни только валидный JSON-массив из 5 элементов."""

print("Начинаем генерацию датасета...")

for i in range(200):
    print(f"Итерация {i+1}/200...")

    try:

        response_text = ask_yandex_gpt(system_prompt_gen, user_prompt_gen, temperature=0.6)
        print('response_text')
        cleaned_text = re.sub(r'^```(json)?\s*', '', response_text.strip(), flags=re.IGNORECASE)
        cleaned_text = re.sub(r'\s*```$', '', cleaned_text).strip()

        tasks = json.loads(cleaned_text)
        for idx, task in enumerate(tasks):
            task['id'] = f"task_iter{i+1}_{idx+1}"

        synthetic_dataset.extend(tasks)
        print(f"  Успешно добавлено {len(tasks)} задач. Всего в датасете: {len(synthetic_dataset)}")

    except json.JSONDecodeError:
        print(f"  Ошибка парсинга JSON. Модель вернула невалидный формат. Пропускаем итерацию.")
    except Exception as e:
        print(f"  Возникла ошибка: {e}")

    time.sleep(1)


with open(save_file, 'w', encoding='utf-8') as f:
    json.dump(synthetic_dataset, f, ensure_ascii=False, indent=2)

print("-" * 40)
print(f"Генерация завершена! Успешно собрано задач: {len(synthetic_dataset)}")

Начинаем генерацию датасета...
Итерация 1/200...
response_text
  Успешно добавлено 5 задач. Всего в датасете: 5
Итерация 2/200...
response_text
  Успешно добавлено 5 задач. Всего в датасете: 10
Итерация 3/200...
response_text
  Успешно добавлено 5 задач. Всего в датасете: 15
Итерация 4/200...
response_text
  Успешно добавлено 5 задач. Всего в датасете: 20
Итерация 5/200...
response_text
  Успешно добавлено 5 задач. Всего в датасете: 25
Итерация 6/200...
response_text
  Успешно добавлено 5 задач. Всего в датасете: 30
Итерация 7/200...
response_text
  Успешно добавлено 5 задач. Всего в датасете: 35
Итерация 8/200...
response_text
  Успешно добавлено 5 задач. Всего в датасете: 40
Итерация 9/200...
response_text
  Успешно добавлено 5 задач. Всего в датасете: 45
Итерация 10/200...
response_text
  Успешно добавлено 5 задач. Всего в датасете: 50
Итерация 11/200...
response_text
  Успешно добавлено 5 задач. Всего в датасете: 55
Итерация 12/200...
response_text
  Успешно добавлено 5 задач. Всег

In [13]:
import requests
import json
from google.colab import userdata

API_KEY = userdata.get('YANDEX_API_KEY')
FOLDER_ID = userdata.get('YANDEX_FOLDER_ID')

def ask_yandex_gpt_arb(system_prompt, user_prompt, temperature=0.3):
    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Api-Key {API_KEY}"
    }
    data = {
        "modelUri": f"gpt://{FOLDER_ID}/yandexgpt/latest",
        "completionOptions": {
            "stream": False,
            "temperature": temperature,
            "maxTokens": 2000
        },
        "messages": [
            {"role": "system", "text": system_prompt},
            {"role": "user", "text": user_prompt}
        ]
    }
    response = requests.post(url, headers=headers, json=data)
    return response.json()['result']['alternatives'][0]['message']['text']

In [14]:
import json
import time
import re

arbiter_results = []

arbiter_system_prompt = """Ты — строгий LLM-арбитр для ревью задач по программированию.
Твоя цель — найти логические ошибки в сгенерированной задаче.
Оцени задачу по трем критериям:
1. Соответствует ли evaluate_code инструкции? Нет ли там неверных assert'ов (например, 2*3=5)? Если да, то значение reason_category ставь eval_mismatch
2. Покрывают ли failure_cases типичные неправильные решения? Не является ли код в failure_cases на самом деле ПРАВИЛЬНЫМ решением? Если да, то значение reason_category ставь poor_coverage
3. Нет ли очевидной невыполнимости или противоречий? Если да, то значение reason_category ставь impossible

Ответь СТРОГО в формате JSON без лишнего текста:
{
  "status": "ok" или "revise",
  "reason_category": "eval_mismatch" | "poor_coverage" | "impossible" | "none",
  "explanation": "Краткое объяснение твоего вердикта на русском языке"
}"""

print("Starting...")

for i, task in enumerate(synthetic_dataset):
    print(f"Проверка задачи {i+1}/{len(synthetic_dataset)} (ID: {task['id']})")

    user_prompt = f"""
    Инструкция: {task['instruction']}
    Код проверки (тесты): {task['evaluate_code']}
    Примеры неправильных решений (failure_cases): {task['failure_cases']}
    """

    try:
        response_text = ask_yandex_gpt_arb(arbiter_system_prompt, user_prompt, temperature=0.1)

        cleaned_text = re.sub(r'^```(json)?\s*', '', response_text.strip(), flags=re.IGNORECASE)
        cleaned_text = re.sub(r'\s*```$', '', cleaned_text).strip()

        evaluation = json.loads(cleaned_text)

        result_record = {
            "task_id": task["id"],
            "instruction": task["instruction"],
            "arbiter_status": evaluation.get("status"),
            "arbiter_reason": evaluation.get("reason_category"),
            "arbiter_explanation": evaluation.get("explanation")
        }
        arbiter_results.append(result_record)
        print(f"Статус: {evaluation.get('status').upper()} | Причина: {evaluation.get('reason_category')}")
        print(f"Объяснение: {evaluation.get('explanation')}")


    except json.JSONDecodeError:
        print(f"Ошибка парсинга ответа Арбитра. Пропускаем.")
    except Exception as e:
        print(f" Ошибка сети или API: {e}")

    time.sleep(1)

arbiter_save_file = f'{drive_path}/arbiter_results.json'
with open(arbiter_save_file, 'w', encoding='utf-8') as f:
    json.dump(arbiter_results, f, ensure_ascii=False, indent=2)

print("Проверка завершена! Результаты успешно сохранены.")

Starting...
Проверка задачи 1/995 (ID: task_iter1_1)
Статус: OK | Причина: none
Объяснение: Задача сформулирована корректно, тесты соответствуют инструкции, примеры неправильных решений отражают типичные ошибки.
Проверка задачи 2/995 (ID: task_iter1_2)
Статус: REVISE | Причина: eval_mismatch
Объяснение: В коде проверки (evaluate) ожидаемое количество букв в строке 'Hello, World!' установлено как 12, что неверно, так как в этой строке 12 символов, включая пробелы и запятую, а количество букв — 10.
Проверка задачи 3/995 (ID: task_iter1_3)
Статус: OK | Причина: none
Объяснение: Задача сформулирована корректно, тесты соответствуют инструкции, примеры неправильных решений действительно являются неверными.
Проверка задачи 4/995 (ID: task_iter1_4)
Статус: OK | Причина: none
Объяснение: Задача сформулирована корректно, тесты соответствуют инструкции, а примеры неправильных решений действительно демонстрируют типичные ошибки.
Проверка задачи 5/995 (ID: task_iter1_5)
Статус: OK | Причина: none
О

In [15]:
import json
import pandas as pd


arbiter_save_file = '/content/drive/MyDrive/research_proposal/arbiter_results.json'


with open(arbiter_save_file, 'r', encoding='utf-8') as f:
    results = json.load(f)

df = pd.DataFrame(results)

total_tasks = len(df)
rejected_tasks = len(df[df['arbiter_status'] == 'revise'])
rejection_rate = (rejected_tasks / total_tasks) * 100 if total_tasks > 0 else 0

reasons_dist = df[df['arbiter_status'] == 'revise']['arbiter_reason'].value_counts()
top_reason = reasons_dist.index[0] if not reasons_dist.empty else "N/A"
top_reason_count = reasons_dist.iloc[0] if not reasons_dist.empty else 0
top_reason_pct = (top_reason_count / rejected_tasks) * 100 if rejected_tasks > 0 else 0

print(f"Всего задач проверено: {total_tasks}")
print(f"Отбраковано Арбитром: {rejected_tasks}")
print(f"Rejection Rate: {rejection_rate:.1f}%")

for reason, count in reasons_dist.items():
    pct = (count / rejected_tasks) * 100
    print(f" - {reason}: {count} задач ({pct:.1f}%)")



Всего задач проверено: 995
Отбраковано Арбитром: 216
Rejection Rate: 21.7%
 - poor_coverage: 125 задач (57.9%)
 - eval_mismatch: 91 задач (42.1%)
